# Day4: Differential Expression

このノートブックでは、差次的発現解析で出てくる基本指標を読み解きます。

このトピックが役立つ場面:
炎症刺激の前後で、どの遺伝子が本当に変化したのかを候補として絞り込みたい場面です。

このトピックで解く課題:
刺激によって本当に変化した遺伝子はどれかを、fold change と有意性の両方を見ながら判断します。

## キーワード辞典（この章）

| キーワード | まずこう理解する | この章での使いどころ | よくある誤解 |
|---|---|---|---|
| differential expression | 条件間で発現が変わる遺伝子探索 | 主要な比較結果 | fold changeだけで判定する |
| log2 fold change | 変化量の対数指標 | 生物学的インパクトの目安 | 有意性を表す指標と混同する |
| p-value | 偶然でこの差が出る確率の指標 | 差の信頼性評価 | 小さい=重要度が高いと単純化 |
| FDR / padj | 多重検定補正後の有意性 | DEG候補の閾値設定 | 補正後値を見ない |
| volcano plot | 変化量と有意性の同時可視化 | 候補遺伝子の俯瞰 | 軸の意味を誤読する |



## この章での判断軸

1. `大きい変化量` と `十分な有意性` を同時に満たすか見る。
2. 閾値は目的に応じて明示し、固定値として盲信しない。
3. 上位遺伝子は次章で機能解釈までつなげる。


## 差次的発現解析で見る2つの軸

差次的発現解析では、遺伝子を2つの観点で評価します。

- 変化量: どれくらい大きく変わったか
- 信頼性: その変化は偶然だけで説明しにくいか

変化量だけを見ると、ばらつきが大きい遺伝子を過大評価することがあります。p-valueだけを見ると、変化量が小さいが統計的には有意な遺伝子を重要そうに見てしまうことがあります。そのため、両方を同時に見る必要があります。

## 差次的発現解析とは

差次的発現解析は、control と treated の間で、どの遺伝子の発現がどの程度変わったかを統計的に評価する工程です。

ここでは本格的な統計モデルではなく、結果の表をどう読むかに焦点を当てます。

In [ ]:
# 差次的発現解析の結果表を読み、volcano plotとして可視化します。
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# DEGテーブルの例です。
# 実務ではDESeq2やedgeRなどのツールから、似た形式の表が出力されます。
deg = pd.DataFrame({
    "gene": ["TP53", "MYC", "GAPDH", "ACTB", "IL6"],
    "log2FoldChange": [1.2, 1.1, 0.1, 0.0, 2.8],
    "pvalue": [0.04, 0.03, 0.80, 0.95, 0.0005],
    "padj": [0.08, 0.06, 0.90, 0.95, 0.002]
}).set_index("gene")

deg

## 重要な列

- `log2FoldChange`: 発現変化の大きさと方向
- `pvalue`: その差が偶然だけで起きたと考えたときの確率
- `padj`: 多重検定補正後の値

`log2FoldChange` が正なら treated で高く、負なら treated で低いことを示します。

## 多重検定という問題

RNA-seqでは、1つの遺伝子だけでなく、数千から数万の遺伝子を同時に調べます。検定を何度も行うと、偶然に有意に見える遺伝子が出やすくなります。

そのため、`pvalue` だけでなく、補正済みの `padj` を見ることが重要です。`padj` は、多数の遺伝子を同時に調べていることを考慮した値です。

In [ ]:
# adjusted p-value が小さいほど、-log10(padj) は大きくなります。
# volcano plot の縦軸として使うことで、有意性を見やすくします。
deg["minus_log10_padj"] = -np.log10(deg["padj"])
deg.sort_values(["padj", "log2FoldChange"])

## Volcano plot

Volcano plot は、横軸に発現変化の大きさ、縦軸に統計的な強さを置いた図です。右上にある遺伝子ほど「大きく上昇し、しかも有意」であることを示します。

## Volcano plot の読み方

Volcano plot は、差次的発現解析の結果を一枚で眺めるための図です。

- 右側: treatedで高い遺伝子
- 左側: treatedで低い遺伝子
- 上側: 統計的に信頼しやすい遺伝子
- 中央付近: 変化量が小さい遺伝子

右上にある遺伝子は、変化量が大きく、統計的にも目立つため、解釈の候補になります。この教材では `IL6` がその例です。

In [ ]:
# 横軸は変化量、縦軸は統計的な強さです。
# 右上にある遺伝子ほど「大きく上昇し、有意性も高い」と読めます。
plt.figure(figsize=(5, 4))
plt.scatter(deg["log2FoldChange"], deg["minus_log10_padj"], color="#4C78A8", s=70)
for gene, row in deg.iterrows():
    plt.text(row["log2FoldChange"] + 0.03, row["minus_log10_padj"] + 0.03, gene)
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("log2 fold change")
plt.ylabel("-log10 adjusted p-value")
plt.title("Volcano plot")
plt.tight_layout()
plt.show()

この例では `IL6` が右上に位置し、treated で強く上昇していて、統計的にも目立つ遺伝子だと読めます。`GAPDH` や `ACTB` は大きな変化がなく、比較的安定です。

## ここまでの要点

- 差次的発現解析では「どれだけ変わったか」と「どれくらい信頼できるか」を同時に見る
- `log2FoldChange` は変化量、`padj` は有意性の目安
- Volcano plot はその二つを一枚で見せる
- 次は遺伝子リストを生物学的意味へつなげる